# 03 — Train the Model

**Pipeline:** Dataset → EDA → Preprocessing → Augmentation → **Training** → Evaluation → Error Analysis → Inference → Deployment

Training is one call — the loop (AMP, cosine LR, early stopping, checkpointing) lives in
[`src/train.py`](../src/train.py), so the exact same run works from the terminal:

```bash
python -m src.train --config configs/config.yaml
```

Transfer learning is the default: an ImageNet-pretrained backbone with a fresh head.
On a small dataset, start with `freeze_backbone: true` (feature extraction), then
unfreeze for a second run if val accuracy plateaus.

In [ ]:
# run from notebooks/ -- make the repo root importable and the cwd
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)

from src.utils import load_config, seed_everything
cfg = load_config("configs/config.yaml")
seed_everything(cfg["seed"])
print("project:", cfg["project_name"], "| data root:", cfg["data"]["root"])

## What are we about to train?

In [ ]:
from src.dataset import build_datasets
from src.model import build_model, count_parameters

classes = build_datasets(cfg)["classes"]
model = build_model(cfg, num_classes=len(classes))
p = count_parameters(model)
print(f"arch: {cfg['model']['arch']} | pretrained: {cfg['model']['pretrained']} | frozen backbone: {cfg['model']['freeze_backbone']}")
print(f"parameters: {p['total']:,} total | {p['trainable']:,} trainable ({p['trainable_pct']:.1f}%)")
del model

## Train

Progress prints per epoch; `best.pt` is saved whenever val accuracy improves, so an interrupted run still leaves a usable model.

In [ ]:
from src.train import train

result = train(cfg)
print("\nbest val accuracy:", f"{result['best_val_acc']:.3f}", "at epoch", result["best_epoch"])

## Learning curves

Read them like this: **gap widening** → overfitting (augment more / stop earlier). **Both flat and high loss** → underfitting (unfreeze, train longer, raise LR). **Val bouncing hard** → LR too high or val split too small.

In [ ]:
import csv
import matplotlib.pyplot as plt
from pathlib import Path

with open(Path(cfg["output"]["dir"]) / "history.csv", encoding="utf-8") as f:
    hist = list(csv.DictReader(f))
epochs = [int(h["epoch"]) for h in hist]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, [float(h["train_loss"]) for h in hist], label="train")
ax1.plot(epochs, [float(h["val_loss"]) for h in hist], label="val")
ax1.set_title("loss"); ax1.set_xlabel("epoch"); ax1.legend()
ax2.plot(epochs, [float(h["train_acc"]) for h in hist], label="train")
ax2.plot(epochs, [float(h["val_acc"]) for h in hist], label="val")
ax2.set_title("accuracy"); ax2.set_xlabel("epoch"); ax2.legend()
plt.tight_layout(); plt.show()

## Experiments worth running

Each is a one-line YAML change — rerun this notebook and compare `best_val_acc`:

1. `model.freeze_backbone: true` — how far does feature extraction alone get you?
2. `model.arch: resnet50` — does capacity help, or just cost time?
3. `model.arch: mobilenet_v3_small` — how much accuracy does a deploy-friendly model give up?

Next: **[04 — Evaluate the Model](04_evaluate_model.ipynb)** — on the *test* split the training never saw.